# 04 · AIPW Evaluation & Calibration

**Purpose:** Compute AIPW pseudo-outcomes across attempts and non-attempts for evaluation.

**Inputs:** Reference/pbp_head.csv, reports/attempt_p_hat_sample.csv, reports/fg_success_calibration.csv

**Outputs:** reports/aipw_calibration.csv, reports/aipw_reliability_plot.png

- [Parameters & Modes](#parameters--modes)
- [Imports](#imports--install-if-missing)
- [Utilities & Helpers](#utilities--helpers-≤40-lines-each)
- [Data Load & Peek](#data-load--peek)
- [Stage Logic](#stage-logic)
- [Artifacts](#artifacts)
- [Session Info](#session-info)


In [ ]:
# Parameters & Modes
SMOKE_MODE <- TRUE
FULL_MODE <- !SMOKE_MODE

reference_dir <- 'Reference'
data_dir <- 'data'
reports_dir <- 'reports'
config_path <- file.path('config', 'params.yaml')

if (!dir.exists(reports_dir)) {
  dir.create(reports_dir, recursive = TRUE)
}

params <- list(
  time_knots = c(60, 120, 300),
  p_clip_min = 0.05,
  p_clip_max = 0.95,
  tau_grid = c(0.03, 0.05, 0.07, 0.10),
  distance_cap = 65,
  yardline_spline_df = 5,
  weight_floor = 0.1,
  weight_cap = 10,
  late_game_threshold = 120,
  distance_spline_df = 6,
  wind_bins = c(0, 5, 10, 15, 25),
  default_p_hat = 0.5,
  default_m_hat = 0.65,
  overall_success_rate = 0.85
)

if (file.exists(config_path)) {
  tryCatch({
    config_params <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, config_params, keep.null = TRUE)
  }, error = function(e) message('Config read failed, using defaults: ', e$message))
}

list2env(params, envir = .GlobalEnv)
set.seed(101)


In [ ]:
# Imports — install if missing
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'mgcv', 'splines', 'glmmTMB', 'pROC', 'yaml', 'scales'
)

install_if_missing <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
  }
}

invisible(purrr::walk(dependencies, install_if_missing))

library(dplyr)
library(tibble)
library(tidyr)
library(readr)
library(stringr)
library(purrr)
library(ggplot2)
library(mgcv)
library(splines)
library(glmmTMB)
library(pROC)
library(yaml)
library(scales)


### Function Index
- `get_schema()` — quick schema glance at data frames.
- `add_time_features()` — derive score/time convenience features.
- `ensure_columns()` — add fallback columns with default values.
- `clip_weights()` — enforce weight floor/cap.
- `stabilize_weights()` — compute stabilized attempt weights.
- `calc_brier()` — calculate (weighted) Brier score.
- `plot_calibration()` — convenience calibration scatter/smoother.


In [ ]:
# Utilities & Helpers (≤40 lines each)
get_schema <- function(df, n = 5) {
  tibble::tibble(
    name = names(df),
    class = purrr::map_chr(df, ~ paste(class(.x), collapse = '/')),
    example = purrr::map_chr(df, ~ paste(head(.x, n), collapse = ', '))
  )
}

add_time_features <- function(df) {
  df %>%
    mutate(
      score_diff = dplyr::coalesce(score_diff, score_differential, 0),
      time_remaining = dplyr::coalesce(game_seconds_remaining, quarter_seconds_remaining, 0),
      log_time_remaining = log1p(time_remaining),
      late_game = time_remaining <= late_game_threshold,
      one_score = abs(score_diff) <= 8
    )
}

ensure_columns <- function(df, defaults) {
  for (nm in names(defaults)) {
    if (!nm %in% names(df)) {
      df[[nm]] <- defaults[[nm]]
    }
  }
  df
}

clip_weights <- function(w, floor = weight_floor, cap = weight_cap) {
  pmin(pmax(w, floor), cap)
}

stabilize_weights <- function(p_hat, base_rate) {
  clip_weights(base_rate / p_hat)
}

calc_brier <- function(actual, predicted, weights = NULL) {
  if (is.null(weights)) {
    mean((predicted - actual) ^ 2)
  } else {
    sum(weights * (predicted - actual) ^ 2) / sum(weights)
  }
}

plot_calibration <- function(df, prob_col, outcome_col, group_col, path) {
  plot <- ggplot(df, aes_string(x = prob_col, y = outcome_col, color = group_col)) +
    geom_point(alpha = 0.4) +
    geom_smooth(method = 'loess', se = FALSE) +
    labs(title = 'Calibration', x = 'Predicted', y = 'Observed')
  ggsave(path, plot = plot, width = 6, height = 4, dpi = 150)
  invisible(plot)
}


In [ ]:
# Data Load & Peek
pbp <- readr::read_csv(file.path(reference_dir, 'pbp_head.csv'), show_col_types = FALSE)

pbp <- ensure_columns(pbp, list(
  season = 2015L,
  play_id = '0',
  game_id = '0',
  kick_distance = 35,
  kickable_fourth_down = TRUE,
  success_flag = NA_real_,
  play_type = 'field_goal',
  field_goal_result = 'made'
))

attempt_preds_path <- file.path(reports_dir, 'attempt_p_hat_sample.csv')
fg_model_path <- file.path(reports_dir, 'fg_success_calibration.csv')

attempt_preds <- if (file.exists(attempt_preds_path)) {
  readr::read_csv(attempt_preds_path, show_col_types = FALSE)
} else {
  tibble(game_id = character(), play_id = character(), season = integer(), p_hat = numeric(), w = numeric())
}

fg_calibration <- if (file.exists(fg_model_path)) {
  readr::read_csv(fg_model_path, show_col_types = FALSE)
} else {
  tibble(distance_bin = character(), n = integer(), rate = numeric(), mean_pred = numeric())
}

if (SMOKE_MODE) {
  pbp <- pbp %>% slice_sample(n = min(6000, n()))
}

pbp <- pbp %>% add_time_features()
get_schema(pbp) %>% print(n = 10)


In [ ]:
# Stage Logic — AIPW Evaluation
## TODO: replace placeholder predictions with model-derived metrics.

full_pool <- pbp %>%
  filter(kickable_fourth_down) %>%
  left_join(attempt_preds, by = c('game_id', 'play_id', 'season')) %>%
  mutate(
    success_flag = dplyr::coalesce(success_flag, ifelse(play_type == 'field_goal' & field_goal_result == 'made', 1, NA_real_)),
    A = ifelse(!is.na(success_flag), 1, 0),
    Y = ifelse(A == 1, success_flag, NA_real_),
    p_hat = dplyr::coalesce(p_hat, default_p_hat),
    model_score = ifelse(A == 1 & !is.na(Y), Y, default_m_hat)
  )

pseudo <- full_pool %>%
  mutate(
    aipw = model_score + (A / p_hat) * (ifelse(is.na(Y), model_score, Y) - model_score),
    aipw = pmin(pmax(aipw, 0), 1)
  )

calibration <- pseudo %>%
  mutate(distance_bin = cut(kick_distance, breaks = seq(10, 65, by = 5), include.lowest = TRUE)) %>%
  group_by(distance_bin) %>%
  summarise(
    n = dplyr::n(),
    aipw_mean = mean(aipw),
    .groups = 'drop'
  )

readr::write_csv(calibration, file.path(reports_dir, 'aipw_calibration.csv'))

reliability_plot <- ggplot(pseudo, aes(x = kick_distance, y = aipw)) +
  geom_point(alpha = 0.25) +
  geom_smooth(method = 'loess', se = FALSE) +
  labs(title = 'AIPW Reliability — Smoke Mode', x = 'Distance', y = 'AIPW pseudo-outcome')

ggplot2::ggsave(
  filename = file.path(reports_dir, 'aipw_reliability_plot.png'),
  plot = reliability_plot,
  width = 7,
  height = 4,
  dpi = 150
)

# Artifact note
message('Calculated placeholder AIPW pseudo-outcomes and reliability plot.')


### Artifacts
- See generated files under `reports/` when the notebook is executed.


In [ ]:
# Session Info
info <- capture.output(sessionInfo())
readr::write_lines(info, file.path(reports_dir, 'session_info.txt'), append = TRUE)
cat(info, sep = '
')
